In [0]:
%run ../config/config_init

In [0]:
domain_name = create_widget("domain_name", "")
job_id = create_widget("job_id")
job_run_id = create_widget("job_run_id")
task_run_id = create_widget("task_run_id")
orchestrator_job_id = create_widget("orchestrator_job_id")
orchestrator_job_run_id = create_widget("orchestrator_job_run_id")

domain_name = dbutils.widgets.get("domain_name")
job_id = dbutils.widgets.get("job_id")
job_run_id = dbutils.widgets.get("job_run_id")
task_run_id = dbutils.widgets.get("task_run_id")
orchestrator_job_id = dbutils.widgets.get("orchestrator_job_id")
orchestrator_job_run_id = dbutils.widgets.get("orchestrator_job_run_id")

In [0]:
rows = (
    spark.table(f"{catalog_name}.{schema_config}.configurations_table")
         .filter(f"domain_name = '{domain_name}' AND medallion_layer = 'gold'")
         .collect()
)

for row in rows:
    # Get schema from the target table
    target_schema = spark.table(row.target).schema

    try:
        source_df = spark.table(row.source)

        # Add metadata columns
        source_df = (
            source_df.withColumn("create_date", F.current_timestamp())
                     .withColumn("source_name", F.lit(row.source))
                     .withColumn("meta_job_id", F.lit(job_id))
                     .withColumn("meta_job_run_id", F.lit(job_run_id))
                     .withColumn("meta_task_run_id", F.lit(task_run_id))
                     .withColumn("meta_orchestrator_job_id", F.lit(orchestrator_job_id))
                     .withColumn("meta_orchestrator_job_run_id", F.lit(orchestrator_job_run_id))
        )

    except Exception as e:
        raise Exception(f"Error processing {row.source}: {e}.")

    try:
        # Write to target table
        print(f"Writing to {row.target}...")
        write_table(source_df, row.target, row.operation, row.source_keys)
        print(f"Done writing to {row.target}...")
    except Exception as e:
        raise Exception(f"Error writing to {row.target}: {e}.")